> **VERSIÓN CON SOLUCIONES (uso del profesor)** — incluye implementaciones y respuestas modelo.

# Taller 3 — Redes de información: estructura, ranking y mercados de búsqueda

**MODELOS DE INTERACCIONES SOCIALES — ECON 64597, Sección 1**
Universidad de los Andes · Facultad de Economía · 2026-2
Profesor: Alvaro J. Riascos Villegas · Complementaria: Gabriela Zamora

**Entrega:** 2026-XX-XX · **Peso:** 10% de la nota final
**Grupos:** mínimo 2, máximo 3 personas

**Referencia única: [EK] Easley & Kleinberg (2010), capítulos 13, 14 y 15.**
Texto completo: https://www.cs.cornell.edu/home/kleinber/networks-book/

> **Declaración de uso de IA del profesor.** Este taller —enunciados, celdas de teoría,
> código, casos de prueba y solución— fue elaborado con la asistencia de una herramienta
> de IA (Claude, Anthropic), usada como asistente de redacción e implementación. Todo el
> contenido fue revisado, ejecutado y verificado por el profesor, quien es responsable de
> su corrección y de las ediciones finales. Se les pide a ustedes exactamente lo mismo que
> se aplicó aquí: usar la herramienta si quieren, declararlo, y responder por el resultado.

---

## Objetivo

Los tres capítulos estudian un mismo objeto —la Web entendida como red de información— en
tres niveles de análisis complementarios:

| Cap. | Problema | Resultado central |
|---|---|---|
| 13 | Caracterizar la topología de la Web como grafo dirigido | La estructura de moño: un núcleo fuertemente conexo con regiones de entrada y de salida |
| 14 | Ordenar las páginas a partir de la propia red de enlaces | Dos caracterizaciones de punto fijo: HITS y PageRank |
| 15 | Asignar y tarifar los espacios publicitarios asociados a una consulta | Un mercado de emparejamiento con precios VCG, y su aproximación de uso corriente, GSP |

El elemento común es la **dirección** de las aristas. En el Taller 2 los grafos eran no
dirigidos y el poder de negociación provenía de ocupar un cuello de botella. Aquí la
asimetría entre "quién me enlaza" y "a quién enlazo" determina la estructura global
(cap. 13), define la noción de autoridad (cap. 14) y hace de la publicidad en buscadores
un mercado de dos lados (cap. 15).

El taller consta de ocho ejercicios: dos sobre el capítulo 13 y tres sobre cada uno de los
capítulos 14 y 15. Las celdas de teoría son autocontenidas y siguen la notación del libro,
pero **no reemplazan la lectura**.

**Convención de notación.** $G = (V, E)$ es un grafo **dirigido**; una arista $(u, v)$
significa "$u$ enlaza a $v$". Escribimos $\mathrm{Out}(v)$ para el conjunto de nodos
alcanzables desde $v$ siguiendo aristas hacia adelante, e $\mathrm{In}(v)$ para el
conjunto de nodos desde los cuales se alcanza $v$. Ambos incluyen a $v$.

## Instrucciones de entrega

1. Completen todas las celdas `# TU CÓDIGO AQUÍ` y todas las preguntas
   `TU RESPUESTA AQUÍ`. No borren las celdas de verificación (`assert`).
2. `Kernel → Restart & Run All` en kernel limpio antes de entregar.
   **Un notebook que no corre de principio a fin pierde 20 puntos.**
3. Entreguen un `.zip` autocontenido: notebook ejecutado, `requirements.txt` y un
   `README.md` de media página.
4. Aplica el protocolo de la sección 5.b del programa: auditoría con *AI code reviewer*
   y **micro-reunión** de defensa. *"No sé, lo escribió la IA"* cuenta como entrega
   incompleta.

## Rúbrica (100 puntos)

| Parte | Tema | Ejercicios | Puntos |
|---|---|---|---|
| 1 | La estructura de la Web | 1.1 – 1.2 | 24 |
| 2 | Análisis de enlaces y ranking | 2.1 – 2.3 | 34 |
| 3 | Mercados de búsqueda patrocinada | 3.1 – 3.3 | 32 |
| 4 | Discusión escrita | 4.1 | 10 |


---
## Datos del grupo y Declaración de Uso de IA

El uso de herramientas de IA está **permitido**; lo que se evalúa es el criterio con que
se emplean. La declaración es un requisito de entrega, no un ítem con puntaje: conforme a
la sección 5.a del programa, una entrega sin declaración —o con una declaración genérica—
se considera incompleta.


In [ ]:
GRUPO = {
    "integrantes": ["Nombre Apellido — 202512345", "Nombre Apellido — 202567890"],
    "seccion": 1,
    "fecha_entrega": "2026-XX-XX",
}

DECLARACION_IA = """
Herramientas usadas: (p. ej. Claude Opus 5, ChatGPT, GitHub Copilot; o "ninguna")

Dónde se usaron y para qué:
- Ejercicio X.Y: ...

Qué se verificó, corrigió o rechazó de la salida de la IA y por qué:
- ...

Declaramos que podemos explicar, modificar y defender todo el código entregado.
"""
for i in GRUPO["integrantes"]:
    print(i)

---
## Parte 0 — Preparación

Todos los grafos y todos los mercados están definidos dentro del notebook. No hay descargas
ni archivos externos. Se usa `numpy` para generar instancias aleatorias de prueba y para
comparar vectores; las iteraciones de las Partes 1 y 2 se implementan con diccionarios,
deliberadamente, para que el procedimiento quede explícito.

Las funciones de `networkx` que resuelven un ejercicio directamente
(`strongly_connected_components`, `hits`, `pagerank`) **no pueden usarse dentro de las
implementaciones**; aparecen únicamente en las celdas de verificación, como referencia
contra la cual contrastar el resultado propio.


In [ ]:
import itertools

import numpy as np
import networkx as nx

print("networkx", nx.__version__, "| numpy", np.__version__)

def ok(mensaje):
    print("OK -", mensaje)

---
# Parte 1 — La estructura de la Web ([EK] cap. 13) · 24 puntos

## 1.0 Teoría

### La Web es un grafo *dirigido*

Un hipervínculo tiene un sentido: la página $u$ decide enlazar a $v$, sin que $v$
intervenga ni quede enlazada de vuelta. Esa asimetría, ausente en el Taller 2, es la que
organiza el análisis del capítulo 13. En un grafo no dirigido "estar conectado" es
una relación de equivalencia y la red se parte en componentes conexas. En un grafo
dirigido, en cambio, "yo llego a ti" y "tú llegas a mí" son proposiciones distintas, y de
esa distinción se deriva toda la estructura del capítulo.

Dos conjuntos organizan el análisis. Para un nodo $v$:

$$\mathrm{Out}(v) = \{w : \text{hay un camino dirigido } v \to \cdots \to w\}, \qquad
  \mathrm{In}(v)  = \{u : \text{hay un camino dirigido } u \to \cdots \to v\}.$$

Ambos contienen a $v$ (el camino vacío). $\mathrm{Out}(v)$ se calcula con un recorrido
(BFS o DFS) sobre $G$ desde $v$; $\mathrm{In}(v)$ es exactamente lo mismo sobre el grafo
**reverso** $G^{R}$, el que resulta de voltear todas las aristas. Esa observación —una
sola rutina de recorrido sirve para las dos— es la que van a usar en el ejercicio 1.1.

### Componentes fuertemente conexas

> **Componente fuertemente conexa (CFC).** Un conjunto maximal de nodos tal que cualquiera
> de ellos alcanza a cualquier otro por caminos dirigidos.

La caracterización operativa es inmediata y es la que conviene implementar:

$$\mathrm{CFC}(v) = \mathrm{Out}(v) \cap \mathrm{In}(v).$$

En efecto, $w$ está en la componente de $v$ si y solo si $v$ llega a $w$ **y** $w$ llega a
$v$. De aquí sale también que las CFC **particionan** $V$: "alcanzarse mutuamente" es
reflexiva, simétrica y transitiva, luego es una relación de equivalencia y sus clases son
las componentes. Un nodo que no está en ningún ciclo forma él solo una CFC de tamaño 1.

### La condensación es un DAG

Si contraemos cada CFC a un solo supernodo y conservamos las aristas entre supernodos
distintos, obtenemos la **condensación** $G^{\mathrm{SCC}}$. El resultado es siempre
acíclico: si hubiera un ciclo entre dos supernodos $X \neq Y$, todos los nodos de
$X \cup Y$ se alcanzarían mutuamente y $X$, $Y$ no habrían sido maximales. Es decir:
**todo grafo dirigido es un DAG de bloques fuertemente conexos**. Esta es la razón por la
cual la estructura descrita en la sección siguiente tiene la forma que tiene: el moño es,
en esencia, un DAG de tres capas.

### La estructura de moño (*bow-tie*)

Broder et al. (2000) rastrearon del orden de $2 \times 10^{8}$ páginas y encontraron una
CFC gigante que ocupaba cerca de una cuarta parte del total. Fijado un nodo cualquiera $r$
de esa componente gigante, todo lo demás se clasifica por su relación con ella:

| Región | Definición formal | Lectura |
|---|---|---|
| **CFC** | la componente fuertemente conexa gigante | El núcleo navegable: de cualquier página a cualquier otra |
| **IN** | $\mathrm{In}(r) \setminus \mathrm{CFC}$ | Llegan al núcleo pero el núcleo no vuelve a ellas |
| **OUT** | $\mathrm{Out}(r) \setminus \mathrm{CFC}$ | Se llega a ellas desde el núcleo, pero no regresan |
| **TUBOS** | fuera de las tres anteriores, alcanzables desde IN y que alcanzan OUT | Atajos de IN a OUT que se saltan el núcleo |
| **TENDRILES** | fuera de las anteriores, pero conectados al resto **ignorando la dirección** | Colgajos que salen de IN o entran a OUT |
| **DESCONECTADOS** | ni siquiera conectados en el grafo no dirigido | Islas |

Dos precisiones importantes:

1. **IN y OUT no dependen del $r$ que se escoja** dentro de la CFC: si $r, r'$ están en la
   misma componente, $\mathrm{Out}(r) = \mathrm{Out}(r')$ e $\mathrm{In}(r) = \mathrm{In}(r')$,
   precisamente porque se alcanzan mutuamente. La construcción está bien definida.
2. La última fila obliga a mirar el grafo **no dirigido** subyacente. Un tendril está
   pegado a la estructura aunque ningún camino dirigido lo conecte con el núcleo.

La conclusión sustantiva del capítulo es negativa: **no existe una "Web navegable" en el
sentido ingenuo del término**. Partiendo de una página cualquiera y siguiendo
enlaces se alcanza a lo sumo la mitad del corpus, y qué mitad se alcanza depende de si el
punto de partida está en IN, en la CFC o en OUT.

## Datos: red de referencia

La red siguiente es de tamaño mínimo, pero contiene las seis regiones. Se recomienda
dibujarla antes de escribir código: hacer explícita la estructura evita la mayor parte de
los errores de implementación.

```
   D → E → [ A ⇄ B ⇄ C ]  →  F → G
       │        (CFC)          ↑
       ├──→ H ─────────────────┘   (tubo: entra por IN, sale a OUT, no toca la CFC)
       └──→ I                      (tendril que sale de IN)
                 J ───────────────┘   (tendril que entra a OUT)
   K → L                             (isla desconectada)
```


In [ ]:
# Red de referencia: aristas (u, v) = "u enlaza a v".
ARISTAS_WEB = [
    ("A", "B"), ("B", "C"), ("C", "A"),   # CFC gigante
    ("D", "E"), ("E", "A"),               # IN
    ("C", "F"), ("F", "G"),               # OUT
    ("E", "H"), ("H", "F"),               # tubo IN -> OUT
    ("E", "I"),                           # tendril que sale de IN
    ("J", "F"),                           # tendril que entra a OUT
    ("K", "L"),                           # isla desconectada
]
WEB = nx.DiGraph(ARISTAS_WEB)

print("nodos:", WEB.number_of_nodes(), "| aristas:", WEB.number_of_edges())
print("grados de salida:", dict(sorted(WEB.out_degree())))

### Ejercicio 1.1 — Alcanzabilidad y componentes fuertemente conexas (12 puntos)

Implementen, **sin usar** `nx.strongly_connected_components` ni
`nx.descendants` / `nx.ancestors`:

- `alcanzables(G, s)`: el conjunto $\mathrm{Out}(s)$, incluyendo a $s$.
- `alcanzan_a(G, s)`: el conjunto $\mathrm{In}(s)$, incluyendo a $s$.
- `cfc_de(G, s)`: la componente fuertemente conexa de $s$.

Sugerencia — Ejercicio 1.1

Escriban **una** rutina de recorrido y úsenla dos veces. Para `alcanzan_a` basta llamarla
sobre `G.reverse(copy=True)`, que devuelve el grafo con todas las aristas volteadas; no
hay que escribir un segundo recorrido "hacia atrás". Un recorrido iterativo con una pila y
un conjunto de visitados evita problemas de recursión: saquen un nodo, miren
`G.successors(u)`, y agreguen los que no hayan visto. `cfc_de` es una línea una vez tienen
las otras dos.

In [ ]:
def alcanzables(G, s):
    """Out(s): nodos alcanzables desde s por caminos dirigidos (incluye a s)."""
    vistos, pila = {s}, [s]
    while pila:
        u = pila.pop()
        for v in G.successors(u):
            if v not in vistos:
                vistos.add(v)
                pila.append(v)
    return vistos


def alcanzan_a(G, s):
    """In(s): nodos desde los cuales se alcanza s (incluye a s)."""
    return alcanzables(G.reverse(copy=True), s)


def cfc_de(G, s):
    """Componente fuertemente conexa de s: Out(s) ∩ In(s)."""
    return alcanzables(G, s) & alcanzan_a(G, s)


for v in ["A", "D", "F", "H", "K"]:
    print(f"  {v}: |Out| = {len(alcanzables(WEB, v))}, |In| = {len(alcanzan_a(WEB, v))}, "
          f"CFC = {sorted(cfc_de(WEB, v))}")

In [ ]:
# Alcanzabilidad en la red de referencia
assert alcanzables(WEB, "A") == {"A", "B", "C", "F", "G"}
assert alcanzan_a(WEB, "A") == {"A", "B", "C", "D", "E"}
assert alcanzables(WEB, "K") == {"K", "L"} and alcanzan_a(WEB, "K") == {"K"}

# La CFC coincide con la de networkx, y no depende del representante
cfcs_nx = {frozenset(c) for c in nx.strongly_connected_components(WEB)}
assert {frozenset(cfc_de(WEB, v)) for v in WEB} == cfcs_nx
assert cfc_de(WEB, "A") == cfc_de(WEB, "B") == cfc_de(WEB, "C") == {"A", "B", "C"}
assert cfc_de(WEB, "H") == {"H"}, "Un nodo sin ciclo es su propia CFC"

# Las CFC particionan los nodos: mismo total, sin traslapes
piezas = {frozenset(cfc_de(WEB, v)) for v in WEB}
assert sum(len(p) for p in piezas) == WEB.number_of_nodes()

# Prueba de estrés contra networkx en un grafo dirigido aleatorio
Gr = nx.gnp_random_graph(45, 0.04, seed=13, directed=True)
assert {frozenset(cfc_de(Gr, v)) for v in Gr} == {frozenset(c)
                                                 for c in nx.strongly_connected_components(Gr)}
ok("Ejercicio 1.1")

### Ejercicio 1.2 — La estructura de moño (12 puntos)

Implementen `estructura_bowtie(G)`, que devuelva un diccionario con las seis regiones de
la tabla de la sección 1.0:

```python
{"CFC": {...}, "IN": {...}, "OUT": {...},
 "TUBOS": {...}, "TENDRILES": {...}, "DESCONECTADOS": {...}}
```

Reglas:

- La **CFC** es la componente fuertemente conexa **más grande** (aquí sí pueden usar
  `nx.strongly_connected_components` para escogerla, o su `cfc_de` del ejercicio anterior).
- **IN** $= \mathrm{In}(r) \setminus \mathrm{CFC}$ y **OUT** $= \mathrm{Out}(r) \setminus \mathrm{CFC}$,
  con $r$ cualquier nodo de la CFC.
- Del resto: un nodo es **tubo** si es alcanzable desde algún nodo de IN *y* alcanza algún
  nodo de OUT; es **desconectado** si no está en la componente conexa de $r$ del grafo **no
  dirigido**; y en cualquier otro caso es **tendril**.
- Las seis regiones deben ser disjuntas y cubrir todos los nodos.

Sugerencia — Ejercicio 1.2

Para "alcanzable desde algún nodo de IN" no hace falta recorrer una vez por cada nodo de
IN: basta con que `alcanzan_a(G, v)` intersecte a IN. Simétricamente, "alcanza OUT" es que
`alcanzables(G, v)` intersecte a OUT. Para la componente débil usen
`nx.node_connected_component(G.to_undirected(), r)`.

In [ ]:
def estructura_bowtie(G):
    """Las seis regiones del moño, como conjuntos disjuntos que cubren todos los nodos."""
    cfc = max(nx.strongly_connected_components(G), key=len)
    r = next(iter(cfc))
    out = alcanzables(G, r) - cfc
    entra = alcanzan_a(G, r) - cfc
    resto = set(G) - cfc - out - entra

    debil = nx.node_connected_component(G.to_undirected(), r)
    tubos, tendriles, desconectados = set(), set(), set()
    for v in resto:
        if v not in debil:
            desconectados.add(v)
        elif (alcanzan_a(G, v) & entra) and (alcanzables(G, v) & out):
            tubos.add(v)
        else:
            tendriles.add(v)

    return {"CFC": cfc, "IN": entra, "OUT": out,
            "TUBOS": tubos, "TENDRILES": tendriles, "DESCONECTADOS": desconectados}


bowtie = estructura_bowtie(WEB)
for region, nodos in bowtie.items():
    print(f"  {region:15s} {sorted(nodos)}")

In [ ]:
assert bowtie["CFC"] == {"A", "B", "C"}
assert bowtie["IN"] == {"D", "E"}
assert bowtie["OUT"] == {"F", "G"}
assert bowtie["TUBOS"] == {"H"}, "H va de IN a OUT sin tocar la CFC"
assert bowtie["TENDRILES"] == {"I", "J"}
assert bowtie["DESCONECTADOS"] == {"K", "L"}

# Es una partición
regiones = list(bowtie.values())
assert set().union(*regiones) == set(WEB)
assert sum(len(x) for x in regiones) == WEB.number_of_nodes(), "Las regiones se traslapan"

# No depende del representante escogido dentro de la CFC
for r in bowtie["CFC"]:
    assert alcanzables(WEB, r) - bowtie["CFC"] == bowtie["OUT"]
    assert alcanzan_a(WEB, r) - bowtie["CFC"] == bowtie["IN"]

# Nadie en OUT regresa a la CFC, y nadie de la CFC llega a IN
assert all(not (alcanzables(WEB, v) & bowtie["CFC"]) for v in bowtie["OUT"])
assert all(not (alcanzables(WEB, "A") & {v}) for v in bowtie["IN"])
ok("Ejercicio 1.2")

**Interpretación 1.2 (obligatoria).** Respondan en 4–5 frases:

1. Un usuario empieza en una página de **IN** y navega solo siguiendo enlaces. ¿Qué
   fracción de esta red alcanza? ¿Y si empieza en **OUT**? Den los dos números exactos
   usando la salida de arriba.
2. En el estudio de Broder et al. la CFC gigante era cerca de un cuarto de las páginas
   rastreadas. ¿Por qué eso implica que la Web **no** es navegable en el sentido ingenuo
   de "desde cualquier página llego a cualquier otra"?
3. ¿Qué le pasaría a la clasificación si agregaran la arista $(G, A)$? Nombren al menos
   dos nodos que cambian de región y expliquen por qué.

> **Respuesta esperada.**

1. Desde una página de IN se alcanza todo menos los tendriles ajenos y la isla: por
   ejemplo desde $D$ se llega a $\{D, E, A, B, C, F, G, H, I\}$, es decir 9 de los 12 nodos
   (75%). Desde OUT, en cambio, $\mathrm{Out}(F) = \{F, G\}$: 2 de 12 nodos (17%). La misma
   red, vista desde dos puntos de partida, tiene tamaños alcanzables que difieren por un
   factor de más de cuatro.
2. Porque "navegable" significaría una sola CFC gigante que contuviera casi todo. Con la
   CFC en torno a un cuarto del corpus, un navegante que arranque en la CFC alcanza a lo
   sumo CFC ∪ OUT (aproximadamente la mitad), uno que arranque en OUT queda atrapado en
   OUT, y uno que arranque en un tendril o en la parte desconectada puede no alcanzar
   nunca el núcleo. La conectividad *dirigida* es mucho más frágil que la no dirigida: el
   mismo corpus es casi todo un solo bloque si se ignoran las direcciones.
3. La arista $(G, A)$ devuelve el flujo de OUT al núcleo, así que $F$ y $G$ pasan a
   alcanzar la CFC y a ser alcanzados desde ella: la CFC gigante crece a
   $\{A, B, C, F, G\}$ y OUT queda vacío. En cascada, $H$ deja de ser tubo (ahora alcanza
   la CFC pero no es alcanzado desde ella: pasa a IN) y $J$ deja de ser tendril por la
   misma razón. Esto ilustra que las regiones no son propiedades locales de un
   nodo, sino del patrón global de ciclos de la red.

---
# Parte 2 — Análisis de enlaces y ranking ([EK] cap. 14) · 34 puntos

## 2.0 Teoría

### El problema

Una consulta como *"newspapers"* devuelve millones de páginas que contienen la palabra.
El texto no alcanza para ordenarlas: todas dicen lo mismo. La idea del capítulo 14 es
usar la **red de enlaces** como un sistema de votación implícito — un enlace de $u$ a $v$
es un voto de $u$ por $v$ — con una calificación esencial: **los votos no tienen todos el
mismo peso**, y el peso de cada voto debe deducirse de la propia red. Esa aparente
circularidad ("una página es buena si la enlazan páginas buenas") no es un defecto lógico:
define un **punto fijo**, y las dos secciones siguientes presentan dos maneras de
caracterizarlo y calcularlo.

### Hubs y autoridades (HITS)

El capítulo distingue dos papeles que una página puede jugar:

- una **autoridad** es un buen destino (la página de un periódico);
- un **hub** es un buen catálogo, una lista de enlaces hacia buenos destinos.

Cada papel se define en términos del otro, y el algoritmo simplemente alterna:

$$\text{(Authority-Update)}\quad a(v) \leftarrow \sum_{u \to v} h(u), \qquad
  \text{(Hub-Update)}\quad h(v) \leftarrow \sum_{v \to w} a(w).$$

Se inicializan todos los valores en 1 y se repite $k$ veces. Sin normalizar, las
magnitudes crecen sin cota; por eso al final de cada ronda se **normaliza** dividiendo
cada vector por la suma de sus entradas. La normalización no cambia el orden relativo — que es lo único que
importa para un ranking — y hace que la sucesión converja.

¿Por qué converge? Si $M$ es la matriz de adyacencia ($M_{uv} = 1$ si $u \to v$), una
ronda completa es $a \leftarrow M^{\top} M \, a$ y $h \leftarrow M M^{\top} h$. Iterar una
matriz simétrica semidefinida positiva y normalizar es el **método de la potencia**: el
límite es el vector propio asociado al valor propio dominante de $M^{\top}M$
(respectivamente $MM^{\top}$). Es decir, HITS calcula un vector propio, y las rondas son
solo la manera transparente de llegar a él.

### PageRank básico

PageRank cambia la metáfora: en vez de dos papeles, un solo número, y en vez de sumar
votos, **repartir** el prestigio propio. Cada página empieza con rango $1/n$ y en cada
ronda entrega **todo** su rango actual dividido en partes iguales entre las páginas que
enlaza:

$$r_{t+1}(v) = \sum_{u \to v} \frac{r_t(u)}{d^{+}(u)}, \qquad d^{+}(u) = \text{grado de salida de } u.$$

La masa total se conserva: $\sum_v r_t(v) = 1$ para todo $t$, así que esto es multiplicar
repetidamente por una matriz **estocástica** y el vector límite es un punto fijo
$r = P^{\top} r$. Un enlace proveniente de una página muy enlazada tiene un
peso alto; uno proveniente de una página que enlaza a mil sitios tiene un peso bajo. Ambos
efectos se siguen de la misma fórmula.

> **Supuesto.** Tal como está escrito, PageRank básico exige que **toda página tenga al
> menos un enlace saliente** — si no, la masa desaparece. Todas las redes de esta parte lo
> cumplen.

### Trampas de rango y el ajuste de escala

El problema de la versión básica es estructural, y el capítulo 13 ya nos dio el vocabulario
para nombrarlo: si un subconjunto de páginas es **absorbente** (se enlazan entre sí y
ninguna enlaza hacia afuera), toda la masa termina ahí y el resto de la Web converge a
cero. Es una **trampa de rango**: un pedazo de la región OUT se queda con todo el
prestigio, sin importar cuán central sea el núcleo.

La corrección es el **PageRank escalado**, con un factor $s \in (0,1)$ (Google usó
$s = 0.85$):

$$r_{t+1}(v) = \frac{1-s}{n} + s \sum_{u \to v} \frac{r_t(u)}{d^{+}(u)}.$$

Una fracción $s$ del rango se reparte por enlaces y la fracción $1-s$ se reparte
**uniformemente entre todas las páginas**, enlazadas o no. Dos lecturas equivalentes:

- **Paseo aleatorio.** Un navegante que en cada paso, con probabilidad $s$, sigue un enlace
  al azar de la página en que está, y con probabilidad $1-s$ **se teletransporta** a una
  página uniformemente al azar. $r$ es la distribución estacionaria de esa cadena de Markov.
- **Álgebra.** La matriz de transición pasa a tener todas sus entradas estrictamente
  positivas, luego la cadena es irreducible y aperiódica. Por Perron–Frobenius el vector
  estacionario **existe, es único y no depende del vector inicial**, y la convergencia es
  geométrica con razón $s$.

El teletransporte es exactamente lo que destruye las trampas: ninguna región puede ser
absorbente si desde cualquier página hay probabilidad positiva de saltar a cualquier otra.

## Datos: dos redes pequeñas

`RED_HITS` tiene tres catálogos (`h1`, `h2`, `h3`) que apuntan a tres destinos
(`a1`, `a2`, `a3`), más un enlace `a1 → a2` que le da a `a1` un papel mixto.
`RED_PR` es fuertemente conexa. `RED_SUMIDERO` contiene la trampa `{A, B}`: `C` y `D`
alimentan al par, y el par no devuelve nada.

In [ ]:
RED_HITS = nx.DiGraph([("h1", "a1"), ("h1", "a2"), ("h1", "a3"),
                       ("h2", "a1"), ("h2", "a2"),
                       ("h3", "a2"),
                       ("a1", "a2")])

RED_PR = nx.DiGraph([("A", "B"), ("A", "C"),
                     ("B", "A"),
                     ("C", "A"), ("C", "D"),
                     ("D", "C"), ("D", "A")])

RED_SUMIDERO = nx.DiGraph([("A", "B"), ("B", "A"),      # trampa absorbente
                           ("C", "A"), ("C", "D"),
                           ("D", "C")])

for nombre, G in [("RED_HITS", RED_HITS), ("RED_PR", RED_PR), ("RED_SUMIDERO", RED_SUMIDERO)]:
    print(f"{nombre:14s} {G.number_of_nodes()} nodos, {G.number_of_edges()} aristas, "
          f"fuertemente conexa: {nx.is_strongly_connected(G)}")

### Ejercicio 2.1 — Hubs y autoridades (11 puntos)

Implementen `hits(G, k=50)`, que devuelva la tupla `(hubs, autoridades)` como
diccionarios, siguiendo estrictamente el procedimiento de [EK] §14.2:

1. Inicializar $a(v) = h(v) = 1$ para todo nodo.
2. Repetir $k$ veces: primero Authority-Update, luego Hub-Update con las autoridades
   **recién** actualizadas, y al final normalizar cada vector dividiendo por su suma.

No usen `nx.hits`: se usa en la celda de verificación como oráculo.

Sugerencia — Ejercicio 2.1

`G.predecessors(v)` da los nodos que enlazan a `v` (para las autoridades) y
`G.successors(v)` los que `v` enlaza (para los hubs). Construyan el diccionario nuevo
completo antes de reemplazar el viejo: si actualizan en el sitio, unos nodos usarían
valores de la ronda $t$ y otros los de $t+1$.

In [ ]:
def hits(G, k=50):
    """(hubs, autoridades) normalizados a suma 1, tras k rondas del algoritmo HITS."""
    aut = {v: 1.0 for v in G}
    hub = {v: 1.0 for v in G}
    for _ in range(k):
        aut = {v: sum(hub[u] for u in G.predecessors(v)) for v in G}
        hub = {v: sum(aut[w] for w in G.successors(v)) for v in G}
        suma_a, suma_h = sum(aut.values()), sum(hub.values())
        if suma_a > 0:
            aut = {v: x / suma_a for v, x in aut.items()}
        if suma_h > 0:
            hub = {v: x / suma_h for v, x in hub.items()}
    return hub, aut


hubs, autoridades = hits(RED_HITS)
print("  autoridades:", {v: round(x, 4) for v, x in sorted(autoridades.items())})
print("  hubs       :", {v: round(x, 4) for v, x in sorted(hubs.items())})

In [ ]:
# Los vectores están normalizados
assert abs(sum(autoridades.values()) - 1) < 1e-9 and abs(sum(hubs.values()) - 1) < 1e-9

# El ranking cualitativo: a2 es la mejor autoridad, h1 el mejor hub
assert max(autoridades, key=autoridades.get) == "a2"
assert autoridades["a2"] > autoridades["a1"] > autoridades["a3"]
assert max(hubs, key=hubs.get) == "h1"
assert all(autoridades[v] < 1e-12 for v in ["h1", "h2", "h3"]), "Un hub puro no es autoridad"

# Coincide con networkx (que normaliza igual, por suma)
h_nx, a_nx = nx.hits(RED_HITS, max_iter=500, tol=1e-14)
assert max(abs(autoridades[v] - a_nx[v]) for v in RED_HITS) < 1e-6
assert max(abs(hubs[v] - h_nx[v]) for v in RED_HITS) < 1e-6

# Más rondas no cambian el resultado: el punto fijo ya se alcanzó
h2_, a2_ = hits(RED_HITS, k=200)
assert max(abs(a2_[v] - autoridades[v]) for v in RED_HITS) < 1e-9
ok("Ejercicio 2.1")

### Ejercicio 2.2 — PageRank básico y trampas de rango (11 puntos)

Implementen `pagerank_basico(G, k=100, r0=None)`: $k$ rondas de la regla

$$r_{t+1}(v) = \sum_{u \to v} \frac{r_t(u)}{d^{+}(u)},$$

partiendo del vector uniforme $r_0(v) = 1/n$ cuando `r0` es `None`, o del diccionario
`r0` si se les pasa uno. No normalicen: la regla ya conserva la masa total, y verificar
que efectivamente se conserva es parte del ejercicio.

In [ ]:
def pagerank_basico(G, k=100, r0=None):
    """k rondas de PageRank básico (sin factor de escala)."""
    n = G.number_of_nodes()
    r = dict(r0) if r0 is not None else {v: 1 / n for v in G}
    for _ in range(k):
        nuevo = {v: 0.0 for v in G}
        for u in G:
            grado = G.out_degree(u)
            if grado == 0:
                continue
            for w in G.successors(u):
                nuevo[w] += r[u] / grado
        r = nuevo
    return r


r_conexa = pagerank_basico(RED_PR)
r_trampa = pagerank_basico(RED_SUMIDERO, k=200)
print("  RED_PR       :", {v: round(x, 4) for v, x in sorted(r_conexa.items())})
print("  RED_SUMIDERO :", {v: round(x, 6) for v, x in sorted(r_trampa.items())})

In [ ]:
# La masa se conserva ronda a ronda
for k in [0, 1, 2, 7, 50]:
    assert abs(sum(pagerank_basico(RED_PR, k=k).values()) - 1) < 1e-9

# Punto fijo exacto en la red fuertemente conexa: A = 2/5, B = 1/5, C = 4/15, D = 2/15
esperado = {"A": 0.4, "B": 0.2, "C": 4 / 15, "D": 2 / 15}
assert max(abs(r_conexa[v] - esperado[v]) for v in RED_PR) < 1e-6
assert abs(sum(r_conexa[u] / RED_PR.out_degree(u)
               for u in RED_PR.predecessors("A")) - r_conexa["A"]) < 1e-9, "r debe ser punto fijo"

# Trampa de rango: toda la masa termina en {A, B}, y C y D quedan en cero
assert abs(r_trampa["A"] - 0.5) < 1e-6 and abs(r_trampa["B"] - 0.5) < 1e-6
assert r_trampa["C"] < 1e-6 and r_trampa["D"] < 1e-6

# C y D no son irrelevantes: tienen enlaces, pero la trampa absorbe todo
assert RED_SUMIDERO.in_degree("C") == 1 and RED_SUMIDERO.out_degree("C") == 2
ok("Ejercicio 2.2")

**Interpretación 2.2 (obligatoria).** Respondan en 4–5 frases:

1. En `RED_SUMIDERO`, la página `C` tiene dos enlaces salientes y uno entrante, y aun así
   su PageRank básico converge a 0. Explíquenlo en términos del **flujo** de rango, no de
   la fórmula.
2. Traduzcan el fenómeno al vocabulario del capítulo 13: ¿en qué región del moño está el
   par $\{A, B\}$ respecto del resto de la red, y por qué eso es exactamente lo que hace
   que sea una trampa?
3. ¿Por qué el orden final de `RED_PR` ($A > C > B > D$) no coincide con el orden por
   grado de entrada? Citen un par de nodos donde difieran.

> **Respuesta esperada.**

1. Todo el rango que `C` recibe lo vuelve a repartir, y la mitad se va hacia `A`, de
   donde nunca regresa: `A` y `B` solo se enlazan entre sí. Ronda tras ronda, la fracción
   de masa que queda del lado de $\{C, D\}$ se multiplica por un factor menor que 1, así
   que tiende a cero geométricamente. El problema no es la calidad de `C`: en la versión
   básica el rango es una masa que se conserva, y existe una región absorbente de la cual
   no sale flujo.
2. Tomando como referencia el par $\{A, B\}$, que es la CFC de esta red, el conjunto
   $\{C, D\}$ está en **IN**: llega a la CFC pero la CFC no vuelve a él. Una CFC sin
   aristas salientes es absorbente, y en el DAG de la condensación es un nodo sumidero: el
   flujo entra y no sale. Por eso la trampa es un fenómeno de estructura, no de tamaño.
3. Por grado de entrada, `A` tiene 3 y `C` tiene 2, pero `B` y `D` tienen 1 cada uno y sin
   embargo `B` (0.2) queda por encima de `D` (0.133). La diferencia es la **calidad** del
   enlace entrante: `B` es enlazado por `A`, que es la página con más rango y solo reparte
   entre dos; `D` es enlazado por `C`, que tiene menos rango y también lo divide en dos.
   Es exactamente el punto del capítulo: no cuentan los votos, cuenta de quién vienen.

### Ejercicio 2.3 — PageRank escalado y unicidad (12 puntos)

Implementen `pagerank_escalado(G, s=0.85, k=200, r0=None)` con la regla

$$r_{t+1}(v) = \frac{1-s}{n} + s \sum_{u \to v} \frac{r_t(u)}{d^{+}(u)}.$$

Las verificaciones comprueban tres propiedades distintas, que no deben confundirse:

- que el resultado **coincide con `nx.pagerank`** (mismo modelo, misma convención);
- que **no depende del vector inicial** — se les compara contra una corrida que arranca de
  una distribución aleatoria generada con `numpy`;
- que **la trampa desaparece**: `C` y `D` reciben masa estrictamente positiva.

In [ ]:
def pagerank_escalado(G, s=0.85, k=200, r0=None):
    """k rondas de PageRank escalado: fracción s por enlaces, fracción 1-s uniforme."""
    n = G.number_of_nodes()
    r = dict(r0) if r0 is not None else {v: 1 / n for v in G}
    for _ in range(k):
        nuevo = {v: (1 - s) / n for v in G}
        for u in G:
            grado = G.out_degree(u)
            if grado == 0:
                continue
            for w in G.successors(u):
                nuevo[w] += s * r[u] / grado
        r = nuevo
    return r


pr_trampa = pagerank_escalado(RED_SUMIDERO)
pr_conexa = pagerank_escalado(RED_PR)
print("  RED_SUMIDERO (s=0.85):", {v: round(x, 6) for v, x in sorted(pr_trampa.items())})
print("  RED_PR       (s=0.85):", {v: round(x, 6) for v, x in sorted(pr_conexa.items())})
print("  s -> 1 en la trampa   :",
      {v: round(x, 4) for v, x in sorted(pagerank_escalado(RED_SUMIDERO, s=0.999).items())})

In [ ]:
# Es una distribución de probabilidad
assert abs(sum(pr_trampa.values()) - 1) < 1e-9 and abs(sum(pr_conexa.values()) - 1) < 1e-9

# Coincide con networkx en las dos redes
for G, mio in [(RED_SUMIDERO, pr_trampa), (RED_PR, pr_conexa)]:
    suyo = nx.pagerank(G, alpha=0.85, tol=1e-14, max_iter=500)
    assert max(abs(mio[v] - suyo[v]) for v in G) < 1e-8

# La trampa ya no absorbe todo
assert pr_trampa["C"] > 0.1 and pr_trampa["D"] > 0.08
assert pr_trampa["A"] > pr_trampa["B"] > pr_trampa["C"] > pr_trampa["D"]

# El límite no depende del vector inicial
rng = np.random.default_rng(2026)
x = rng.random(RED_PR.number_of_nodes())
inicio = dict(zip(sorted(RED_PR), (x / x.sum()).tolist()))
assert abs(sum(inicio.values()) - 1) < 1e-12
otro = pagerank_escalado(RED_PR, r0=inicio)
assert max(abs(otro[v] - pr_conexa[v]) for v in RED_PR) < 1e-9

# Cuando s -> 1 se recupera el problema del ejercicio anterior
casi = pagerank_escalado(RED_SUMIDERO, s=0.999, k=3000)
assert casi["C"] < 0.02 and casi["A"] + casi["B"] > 0.95
ok("Ejercicio 2.3")

---
# Parte 3 — Mercados de búsqueda patrocinada ([EK] cap. 15) · 32 puntos

## 3.0 Teoría

### El objeto: espacios de anuncios sobre una consulta

Junto a los resultados que ordena el capítulo 14 aparece una segunda lista, la de los
anuncios, cuyos espacios sí se venden. El modelo del capítulo 15 tiene tres elementos:

- Hay $m$ **espacios** (*slots*), y el espacio $i$ recibe $r_i$ clics por unidad de tiempo,
  con $r_1 > r_2 > \cdots > r_m$. Esas son las **tasas de clic** (*clickthrough rates*), y
  se suponen una propiedad del espacio, no del anuncio que se ponga ahí.
- Hay $n$ **anunciantes**, y el anunciante $j$ obtiene un valor $v_j$ por cada clic. Cada
  anunciante quiere a lo sumo un espacio.

Si el anunciante $j$ queda en el espacio $i$, el valor generado es $r_i \, v_j$. La
estructura del problema es exactamente la del Taller 2: un **mercado de emparejamiento**
bipartito, con la particularidad de que la matriz de valoraciones no es arbitraria sino
que es un **producto** $r_i v_j$.

### Por qué la asignación óptima es trivial aquí

Esa forma de producto tiene una consecuencia fuerte: para maximizar
$\sum_i r_i v_{\sigma(i)}$ basta ordenar. El mejor espacio para el anunciante de mayor
valor, el segundo para el segundo, y así. La razón es la **desigualdad del reordenamiento**:
si $r_1 > r_2$ y $v_x > v_y$, entonces

$$r_1 v_x + r_2 v_y - (r_1 v_y + r_2 v_x) = (r_1 - r_2)(v_x - v_y) > 0,$$

de modo que cualquier asignación con un par "cruzado" mejora al descruzarlo. Un algoritmo
codicioso resuelve en $O(n \log n)$ lo que en el Taller 2 exigía fuerza bruta sobre $n!$
permutaciones. En el ejercicio 3.1 lo van a comprobar contra la fuerza bruta.

El problema relevante no es entonces **quién** ocupa cada espacio, sino **a qué precio**:
los $v_j$ son información privada y, si el vendedor los pregunta directamente, los
anunciantes tienen incentivos a reportarlos de manera estratégica.

### El principio VCG

> **Regla VCG.** Asignar de manera que se maximice el valor total, y cobrarle a cada
> participante el **daño que le causa a los demás**:
>
> $$p_j = \underbrace{\max_{\text{asignaciones sin } j} \sum_{k \neq j} r_{i(k)} v_k}_{\text{lo mejor que podrían lograr los otros si } j \text{ no existiera}}
>  \; - \; \underbrace{\sum_{k \neq j} r_{i(k)} v_k}_{\text{lo que los otros logran con } j \text{ presente}}.$$

Ese pago es una **externalidad**: lo que el resto del mercado pierde por la presencia de
$j$. Nótese que el que ocupa el último espacio útil suele no desplazar a nadie y por lo
tanto paga 0, mientras que el primero paga por haber empujado a todos los demás un
escalón hacia abajo.

> **Teorema ([EK] §15.4).** Bajo la regla VCG, **decir la verdad es una estrategia
> dominante**: para cualquier reporte de los demás, ningún anunciante obtiene un pago neto
> mayor reportando $v'_j \neq v_j$.

La intuición del argumento: al fijar los reportes ajenos, el pago neto de $j$ cuando el
mecanismo lo pone en el espacio $i$ es $r_i v_j - p_j$, y se puede reescribir como el valor
social total menos una cantidad que **no depende del reporte de $j$**. Mentir solo puede
mover a $j$ a un espacio que no maximiza esa expresión. En el ejercicio 3.2 lo van a
verificar numéricamente, barriendo todos los reportes posibles.

### La subasta generalizada de segundo precio (GSP)

Los buscadores no adoptaron VCG sino un mecanismo más sencillo de comunicar: cada
anunciante puja $b_j$, el de mayor puja se lleva el mejor espacio, el segundo el segundo,
y **cada uno paga por clic la puja del siguiente**.

Con un solo espacio, GSP es la subasta de segundo precio de Vickrey y decir la verdad es
dominante. **Con dos o más espacios deja de serlo**, y el ejemplo canónico está en el
ejercicio 3.3: a veces conviene pujar por debajo del propio valor, perder el mejor espacio
y quedarse con el segundo a un precio mucho más bajo. El pago por clic ya no está atado al
propio reporte de manera inocua, y con ello se pierde la dominancia.

Lo que sí admite el mecanismo es un análisis de equilibrio: GSP tiene **múltiples**
equilibrios de Nash en pujas, con ingresos muy distintos para el buscador, y entre ellos
hay uno cuyos pagos coinciden exactamente con los de VCG. Ese resultado es el principal
argumento a favor del mecanismo, y se verifica por búsqueda exhaustiva en el ejercicio 3.3.

## Datos: dos mercados

`TASAS` y `VALORES` son el mercado principal, con tres espacios útiles. `TASAS_GSP` incluye un
tercer espacio con tasa 0 —equivalente a no ser exhibido— que es el que permite exhibir la
falla de veracidad de GSP.

In [ ]:
TASAS = [10, 5, 2]                       # tasas de clic, de mejor a peor espacio
VALORES = {"x": 7, "y": 6, "z": 1}       # valor por clic de cada anunciante

TASAS_GSP = [10, 4, 0]                   # el tercer espacio no recibe clics

print("valor si x va al espacio 1:", TASAS[0] * VALORES["x"])
print("matriz de valores r_i * v_j:")
for i, r in enumerate(TASAS):
    print(f"  espacio {i+1} (r={r:2d}):", {j: r * v for j, v in VALORES.items()})

### Ejercicio 3.1 — Asignación óptima: fuerza bruta contra algoritmo codicioso (10 puntos)

Se les da `valor_total`. Implementen:

- `asignacion_optima_anuncios(tasas, valores)`: devuelve `(valor, asignacion)` por **fuerza
  bruta** sobre todas las permutaciones, donde `asignacion[i]` es el anunciante que ocupa el
  espacio $i$, o `None` si el espacio queda vacío.
- `asignacion_codiciosa(tasas, valores)`: devuelve solo la `asignacion`, ordenando los
  anunciantes por valor descendente.

**Casos de borde.** Puede haber más espacios que anunciantes —los sobrantes quedan en
`None`— o más anunciantes que espacios, en cuyo caso alguno no recibe asignación. La celda
de verificación prueba ambos casos con instancias aleatorias.

In [ ]:
# Función dada: no hay que modificarla.

def valor_total(tasas, valores, asignacion):
    """Suma de r_i * v_j sobre los espacios ocupados."""
    return sum(tasas[i] * valores[j] for i, j in enumerate(asignacion) if j is not None)

In [ ]:
def asignacion_optima_anuncios(tasas, valores):
    """(valor, asignación) óptima por fuerza bruta sobre las permutaciones."""
    anunciantes = list(valores)
    k = min(len(tasas), len(anunciantes))
    mejor_valor, mejor_asig = None, None
    for perm in itertools.permutations(anunciantes, k):
        asig = list(perm) + [None] * (len(tasas) - k)
        valor = valor_total(tasas, valores, asig)
        if mejor_valor is None or valor > mejor_valor:
            mejor_valor, mejor_asig = valor, asig
    return mejor_valor, mejor_asig


def asignacion_codiciosa(tasas, valores):
    """Anunciantes ordenados por valor descendente; espacios sobrantes en None."""
    orden = sorted(valores, key=lambda j: -valores[j])[:len(tasas)]
    return list(orden) + [None] * (len(tasas) - len(orden))


v_opt, asig_opt = asignacion_optima_anuncios(TASAS, VALORES)
print(f"óptima   : {asig_opt} -> valor {v_opt}")
print(f"codiciosa: {asignacion_codiciosa(TASAS, VALORES)}")

In [ ]:
assert asig_opt == ["x", "y", "z"] and v_opt == 10 * 7 + 5 * 6 + 2 * 1 == 102
assert asignacion_codiciosa(TASAS, VALORES) == asig_opt

# Cruzar dos anunciantes siempre empeora: (r1 - r2)(vx - vy) > 0
assert valor_total(TASAS, VALORES, ["y", "x", "z"]) == 97 == v_opt - (10 - 5) * (7 - 6)

# Más espacios que anunciantes: los sobrantes quedan vacíos
assert asignacion_codiciosa([9, 4, 1], {"a": 5}) == ["a", None, None]

# La codicia coincide con la fuerza bruta en 200 instancias aleatorias
rng = np.random.default_rng(15)
for _ in range(200):
    m, n = int(rng.integers(1, 5)), int(rng.integers(1, 6))
    tasas = sorted(rng.integers(0, 20, size=m).tolist(), reverse=True)
    vals = {f"a{i}": int(rng.integers(0, 15)) for i in range(n)}
    assert valor_total(tasas, vals, asignacion_codiciosa(tasas, vals)) == \
           asignacion_optima_anuncios(tasas, vals)[0]
ok("Ejercicio 3.1")

### Ejercicio 3.2 — Precios VCG y veracidad (12 puntos)

Implementen `precios_vcg(tasas, valores)`: un diccionario `{anunciante: pago total}` según
la fórmula del daño causado. Tres precisiones:

- El pago es **total**, no por clic; el precio por clic se obtiene dividiendo por la tasa
  del espacio asignado.
- Un anunciante que no recibe espacio paga 0.
- El máximo "sin $j$" se calcula con el mismo mercado, quitando a $j$ del diccionario de
  valores.

Después se verifica la **veracidad**: para cada anunciante se barre todo un rango de
reportes posibles y se comprueba que ninguno le da un pago neto estrictamente mayor que
decir la verdad.

In [ ]:
def precios_vcg(tasas, valores):
    """{anunciante: pago total} = daño causado a los demás por su presencia."""
    _, optima = asignacion_optima_anuncios(tasas, valores)
    precios = {}
    for j in valores:
        sin_j = {a: v for a, v in valores.items() if a != j}
        mejor_sin_j = asignacion_optima_anuncios(tasas, sin_j)[0] if sin_j else 0
        otros_con_j = sum(tasas[i] * valores[a]
                          for i, a in enumerate(optima) if a is not None and a != j)
        precios[j] = mejor_sin_j - otros_con_j
    return precios


precios = precios_vcg(TASAS, VALORES)
for j, p in precios.items():
    espacio = asig_opt.index(j)
    print(f"  {j}: espacio {espacio+1} (r={TASAS[espacio]:2d}) | pago total {p:5.1f} | "
          f"por clic {p/TASAS[espacio]:4.2f} | pago neto {TASAS[espacio]*VALORES[j] - p:5.1f}")

In [ ]:
# Los tres pagos del mercado principal
assert precios == {"x": 33, "y": 3, "z": 0}
assert abs(precios["x"] / 10 - 3.3) < 1e-12 and abs(precios["y"] / 5 - 0.6) < 1e-12
assert precios["z"] == 0, "El último no desplaza a nadie"

# Todo pago es no negativo y no supera el valor que el anunciante obtiene
for j, p in precios.items():
    assert 0 <= p <= TASAS[asig_opt.index(j)] * VALORES[j] + 1e-9

# Veracidad: ningún reporte mejora el pago neto frente a decir la verdad
def pago_neto(j, reporte, tasas, valores):
    v = dict(valores)
    v[j] = reporte
    _, asig = asignacion_optima_anuncios(tasas, v)
    p = precios_vcg(tasas, v)
    return 0.0 if j not in asig else tasas[asig.index(j)] * valores[j] - p[j]

for j in VALORES:
    neto_verdad = pago_neto(j, VALORES[j], TASAS, VALORES)
    assert all(pago_neto(j, r, TASAS, VALORES) <= neto_verdad + 1e-9 for r in range(0, 16)), \
        f"Mentir nunca debe pagarle a {j}"

# El ingreso del buscador es la suma de las externalidades
assert sum(precios.values()) == 36
ok("Ejercicio 3.2")

### Ejercicio 3.3 — La subasta generalizada de segundo precio (10 puntos)

Implementen:

- `resultado_gsp(pujas, tasas)`: devuelve `(asignacion, precio_por_clic)`, dos diccionarios
  `{anunciante: ...}`. Se ordena por puja descendente (empates: por nombre, para que el
  resultado sea determinista), el $i$-ésimo se lleva el espacio $i$ y paga por clic la puja
  del siguiente en el orden, o 0 si no hay siguiente.
- `es_equilibrio_gsp(pujas, tasas, valores, rejilla)`: `True` si ningún anunciante mejora
  su pago cambiando **solo su propia puja** a algún valor de `rejilla`.

Se les dan `pagos_gsp` e `ingreso_gsp`, que dependen de su `resultado_gsp`.

Con `TASAS_GSP = [10, 4, 0]` y los valores del mercado, se comprueban los tres resultados
centrales de [EK] §15.5–15.6: pujar el valor verdadero **no** constituye un equilibrio,
existen **múltiples** equilibrios con ingresos muy distintos, y uno de ellos reproduce
**exactamente** los pagos VCG.

In [ ]:
def resultado_gsp(pujas, tasas):
    """(asignación, precio por clic) de la subasta generalizada de segundo precio."""
    orden = sorted(pujas, key=lambda j: (-pujas[j], j))
    asignacion, precio = {}, {}
    for i in range(min(len(tasas), len(orden))):
        j = orden[i]
        asignacion[j] = i
        precio[j] = pujas[orden[i + 1]] if i + 1 < len(orden) else 0
    return asignacion, precio


def es_equilibrio_gsp(pujas, tasas, valores, rejilla):
    """True si ningún anunciante mejora desviándose unilateralmente dentro de la rejilla."""
    base = pagos_gsp(pujas, tasas, valores)
    for j in pujas:
        for b in rejilla:
            alterna = dict(pujas)
            alterna[j] = b
            if pagos_gsp(alterna, tasas, valores)[j] > base[j] + 1e-9:
                return False
    return True

In [ ]:
# Funciones dadas: dependen de su resultado_gsp.

def pagos_gsp(pujas, tasas, valores):
    """{anunciante: pago neto} = r_i * (v_j - precio por clic)."""
    asignacion, precio = resultado_gsp(pujas, tasas)
    return {j: (tasas[asignacion[j]] * (valores[j] - precio[j]) if j in asignacion else 0.0)
            for j in pujas}


def ingreso_gsp(pujas, tasas):
    """Ingreso total del buscador."""
    asignacion, precio = resultado_gsp(pujas, tasas)
    return sum(tasas[asignacion[j]] * precio[j] for j in asignacion)


REJILLA = [i * 0.5 for i in range(0, 17)]          # 0, 0.5, ..., 8

print("pujando la verdad (7, 6, 1):", pagos_gsp(VALORES, TASAS_GSP, VALORES),
      "| ingreso", ingreso_gsp(VALORES, TASAS_GSP))
desviacion = {**VALORES, "x": 5}
print("x puja 5 en vez de 7      :", pagos_gsp(desviacion, TASAS_GSP, VALORES),
      "| ingreso", ingreso_gsp(desviacion, TASAS_GSP))
print("¿la verdad es equilibrio? :", es_equilibrio_gsp(VALORES, TASAS_GSP, VALORES, REJILLA))

equilibrios = [dict(zip(["x", "y", "z"], c))
               for c in itertools.product(REJILLA, repeat=3)
               if all(c[i] <= VALORES[j] for i, j in enumerate(["x", "y", "z"]))
               and es_equilibrio_gsp(dict(zip(["x", "y", "z"], c)), TASAS_GSP, VALORES, REJILLA)]
ingresos = sorted(ingreso_gsp(e, TASAS_GSP) for e in equilibrios)
print(f"equilibrios sin sobrepujar: {len(equilibrios)} | "
      f"ingreso entre {ingresos[0]} y {ingresos[-1]}")
print("ingreso VCG con estas tasas:", sum(precios_vcg(TASAS_GSP, VALORES).values()))

In [ ]:
# GSP no es veraz: x gana más pujando por debajo de su valor
assert pagos_gsp(VALORES, TASAS_GSP, VALORES)["x"] == 10 * (7 - 6) == 10
assert pagos_gsp({**VALORES, "x": 5}, TASAS_GSP, VALORES)["x"] == 4 * (7 - 1) == 24
assert not es_equilibrio_gsp(VALORES, TASAS_GSP, VALORES, REJILLA), \
    "Pujar la verdad no es equilibrio en GSP con dos o más espacios"

# Con un solo espacio sí se recupera Vickrey: la verdad es equilibrio
assert es_equilibrio_gsp(VALORES, [10], VALORES, REJILLA)

# Multiplicidad de equilibrios y rango de ingresos
assert len(equilibrios) > 50, "GSP tiene muchos equilibrios, no uno"
assert ingresos[0] < 15 and ingresos[-1] > 45

# Hay un equilibrio que reproduce exactamente los precios por clic de VCG
vcg = precios_vcg(TASAS_GSP, VALORES)
asig_vcg = asignacion_optima_anuncios(TASAS_GSP, VALORES)[1]
por_clic_vcg = {j: vcg[j] / TASAS_GSP[asig_vcg.index(j)]
                for j in vcg if TASAS_GSP[asig_vcg.index(j)] > 0}
assert por_clic_vcg == {"x": 4.0, "y": 1.0}

perfil = {"x": 7, "y": 4, "z": 1}
asignacion, precio = resultado_gsp(perfil, TASAS_GSP)
assert es_equilibrio_gsp(perfil, TASAS_GSP, VALORES, REJILLA)
assert asignacion == {"x": 0, "y": 1, "z": 2}, "La asignación eficiente se preserva"
assert precio["x"] == 4 and precio["y"] == 1, "Mismos precios por clic que VCG"
assert ingreso_gsp(perfil, TASAS_GSP) == sum(vcg.values()) == 44
ok("Ejercicio 3.3")

**Interpretación 3.3 (obligatoria).** Respondan en 4–5 frases:

1. En el ejemplo, `x` gana 10 pujando su valor verdadero (7) y 24 pujando 5. Expliquen el
   mecanismo económico de la desviación: ¿qué gana y qué pierde al bajar la puja?
2. VCG es veraz y GSP no. ¿Por qué entonces la industria adoptó GSP? Den al menos dos
   razones y usen el número que acaban de calcular sobre el ingreso.
3. La asignación en el equilibrio $(7, 4, 1)$ es la misma que la de VCG y la misma que la
   del ejercicio 3.1. ¿Es una coincidencia? Relacionen con el patrón que ya vieron en el
   Taller 2: la teoría fija la eficiencia y deja abierto el reparto.

> **Respuesta esperada.**

1. Al bajar de 7 a 5, `x` cede el primer espacio y con él 6 de los 10 clics, pero su
   precio por clic cae de 6 (la puja de `y`) a 1 (la puja de `z`). Pierde volumen y gana
   margen: pasa de un excedente de $10 \times (7-6) = 10$ a $4 \times (7-1) = 24$. Esa es
   exactamente la falla de GSP: como el precio se hereda del **siguiente** en el orden,
   bajar la puja no solo cambia el espacio sino también el precio que se paga, y en VCG
   ese segundo canal no existe.
2. Primero, es mucho más fácil de explicar y auditar: "pagas lo que puja el siguiente" se
   entiende sin teoría de mecanismos, mientras que el precio VCG exige recalcular un
   óptimo contrafactual por anunciante. Segundo, GSP es históricamente lo que había
   (evolucionó del primer precio) y cambiar de mecanismo obliga a que todos los anunciantes
   reaprendan su estrategia. Y tercero, el costo no es evidente: en el rango de equilibrios
   que encontramos el ingreso va de 10 a 49, y el equilibrio $(7,4,1)$ da exactamente 44,
   el mismo ingreso que VCG — GSP no es uniformemente peor para el buscador, y en varios
   equilibrios es mejor.
3. No se trata de una coincidencia. En los tres casos la asignación eficiente es la misma porque el
   valor social solo depende de quién ocupa qué espacio, y con valoraciones de la forma
   $r_i v_j$ el orden por valor es óptimo cualquiera sea el mecanismo de precios. Lo que
   cambia entre VCG y los distintos equilibrios de GSP es únicamente **el reparto** de esos
   102 (o 94, con las tasas de GSP) puntos de excedente entre anunciantes y buscador —
   el mismo patrón del Taller 2, donde los veinte vectores de precios de equilibrio daban
   todos valoración total 23 y repartos completamente distintos.

---
# Parte 4 — Discusión · 10 puntos

**4.1 (10 puntos).** En **máximo 350 palabras**:

> Los tres capítulos articulan estructura, ranking y mercado. Escriban un texto breve que
> los conecte, apoyándose en **sus propios resultados numéricos**:
>
> (a) ¿Qué propiedad estructural del capítulo 13 invalida al PageRank básico, y cómo la
> corrige el factor de escala? Citen los valores obtenidos para `RED_SUMIDERO` con y sin
> escala.
> (b) HITS y PageRank son dos caracterizaciones de punto fijo sobre la misma red. ¿Qué mide
> cada una que la otra no capta? Den un ejemplo de página que ordenarían de manera distinta.
> (c) El capítulo 15 introduce un mercado paralelo al ranking orgánico. Si el buscador puede
> ordenar los resultados por PageRank y vender los espacios publicitarios por VCG, ¿qué
> justifica mantener las dos listas separadas? Argumenten en términos de incentivos.

> **Respuesta esperada.**

**(a)** La propiedad es la existencia de una componente fuertemente conexa **sin aristas
salientes** — un nodo sumidero en la condensación del capítulo 13. En `RED_SUMIDERO` el par
$\{A,B\}$ es esa componente: el PageRank básico converge a
$A = B = 0.5$, $C = D = 0$, es decir, dos páginas con enlaces reales quedan con rango
exactamente cero. Con $s = 0.85$ el mismo cálculo da
$A \approx 0.416$, $B \approx 0.391$, $C \approx 0.109$, $D \approx 0.084$: el
teletransporte inyecta masa $(1-s)/n$ en cada página en cada ronda, ninguna región puede
ser absorbente, y la cadena resultante es irreducible y aperiódica, de modo que el vector
límite existe, es único y no depende del punto de partida (lo verificamos arrancando de una
distribución aleatoria). Subiendo a $s = 0.999$ se recupera casi exactamente la patología.

**(b)** PageRank mide **prestigio recibido**: un solo número por página, que fluye por los
enlaces entrantes y se diluye entre los salientes. HITS mide **dos papeles distintos**: ser
buen destino (autoridad) y ser buen catálogo (hub); una página puede ser excelente en uno y
nula en el otro. En `RED_HITS`, `h1` es el mejor hub (0.354) y tiene autoridad exactamente
0, porque nadie la enlaza: PageRank la pondría al fondo del ranking mientras que HITS la
pone de primera en su dimensión. La diferencia sustantiva es que HITS reconoce el valor de
una página que solo organiza información ajena, y PageRank no.

**(c)** Porque mezclarlas destruiría la señal que hace valioso al ranking orgánico. Si
pagar mejorara la posición orgánica, el PageRank dejaría de ser un agregado de votos
independientes y pasaría a ser un precio, y el incentivo a comprar enlaces contaminaría la
red misma que se está midiendo. Separar las listas mantiene dos mecanismos con objetivos
distintos —uno que estima relevancia a partir de información que los anunciantes no
controlan, otro que asigna espacios escasos a quien más los valora— y le permite al
buscador cobrar por el segundo sin degradar el primero, que es justamente el activo que
atrae a los usuarios.

---

**Nota de calificación.** Se esperan las tres partes con números propios. Un texto correcto
pero sin cifras de sus propias corridas pierde la mitad del puntaje.

---
### Cierre

La celda siguiente verifica los datos del grupo y la presencia de la Declaración de Uso de
IA. No otorga puntaje: es un requisito de entrega, y un notebook que no la ejecute sin
errores se considera una entrega incompleta.

In [ ]:
# Celda de cierre: no la modifiquen.
assert len(GRUPO["integrantes"]) >= 2, "El grupo debe tener mínimo 2 integrantes"
assert "Nombre Apellido" not in " ".join(GRUPO["integrantes"]), "Actualicen los nombres"
assert len(DECLARACION_IA.strip()) > 120, "La Declaración de Uso de IA está incompleta"
print("Taller 3 completado por:", ", ".join(GRUPO["integrantes"]))
print("Recuerden: Kernel -> Restart & Run All antes de comprimir el .zip")

---
### Antes de entregar — lista de chequeo

- [ ] `Kernel → Restart & Run All` corre sin errores.
- [ ] Las 8 celdas de verificación imprimen `OK`.
- [ ] Las 4 preguntas escritas (1.2, 2.2, 3.3 y 4.1) están respondidas.
- [ ] `GRUPO` y `DECLARACION_IA` completos.
- [ ] `requirements.txt` y `README.md` incluidos en el `.zip`.

### Referencia

Easley, D. y Kleinberg, J. (2010). *Networks, Crowds, and Markets*, capítulos 13, 14 y 15.
Cambridge University Press.
https://www.cs.cornell.edu/home/kleinber/networks-book/

Broder, A. et al. (2000). Graph structure in the Web. *Computer Networks* 33, 309–320.